In [1]:
# -*- coding: utf-8 -*-
"""
FASE 1: AUDITORÍA DE DATOS - VERSIÓN EXPLORATORIA
Este código NO asume nada. Solo EXPLORA la base para saber qué tenemos.

Ejecutar DESPUÉS de haber construido BASE_REGRESIONES.csv
"""

import pandas as pd
import numpy as np

# ============================================================
# 1. CARGAR BASE DE DATOS
# ============================================================

ruta_base = r"C:\Users\Dafne\Documents\GitHub\Base-de-datos-investigacion-Economica\Base24_25\BASE_REGRESIONES.csv"
df = pd.read_csv(ruta_base, encoding="utf-8-sig")

print("="*70)
print("AUDITORÍA FASE 1 - BASE_REGRESIONES.csv")
print("="*70)
print(f"✅ Base cargada: {len(df):,} observaciones, {len(df.columns)} variables")
print(f"✅ Columnas disponibles: {df.columns.tolist()}")
print("")

# ============================================================
# 2. ¿QUÉ VARIABLES FINANCIERAS TENEMOS?
# ============================================================

print("="*70)
print("1. VARIABLES FINANCIERAS DISPONIBLES")
print("="*70)

# Buscar columnas que contengan P558 (módulo de servicios financieros)
cols_financieras = [col for col in df.columns if 'P558' in col or 'credito' in col.lower()]
print(f"\n📊 Columnas financieras encontradas:")
for col in cols_financieras:
    print(f"  • {col}")

# Ver específicamente P558E2 y P558E3
print("\n📊 Buscando P558E2 y P558E3 específicamente:")
if 'P558E2' in df.columns:
    print("  ✅ P558E2 ENCONTRADA en la base")
else:
    print("  ❌ P558E2 NO encontrada en la base")

if 'P558E3' in df.columns:
    print("  ✅ P558E3 ENCONTRADA en la base")
else:
    print("  ❌ P558E3 NO encontrada en la base")

# ============================================================
# 3. SI EXISTEN, ¿QUÉ VALORES TIENEN P558E2 y P558E3?
# ============================================================

print("\n" + "="*70)
print("2. ANÁLISIS DE P558E2 Y P558E3 (SI EXISTEN)")
print("="*70)

if 'P558E2' in df.columns:
    print("\n📊 P558E2 - Solicitud de crédito:")
    print(f"  • Tipo de dato: {df['P558E2'].dtype}")
    print(f"  • Valores únicos: {sorted(df['P558E2'].unique())}")
    print(f"  • Value counts:")
    print(df['P558E2'].value_counts(dropna=False).sort_index())
    
    # Convertir a numérico para estadísticas
    df['P558E2_num'] = pd.to_numeric(df['P558E2'], errors='coerce')
    print(f"\n  • Tasa de missing: {df['P558E2_num'].isna().mean():.2%}")
    print(f"  • Tasa de 'Sí' (valor 1): {(df['P558E2_num'] == 1).mean():.2%}")
    print(f"  • Tasa de 'No' (valor 2): {(df['P558E2_num'] == 2).mean():.2%}")
else:
    print("\n❌ P558E2 NO está en la base.")
    print("   → Opciones:")
    print("      a) Agregarla al script de construcción (recomendado)")
    print("      b) Eliminar la afirmación sobre solicitud del documento")

if 'P558E3' in df.columns:
    print("\n📊 P558E3 - Obtención de crédito:")
    print(f"  • Tipo de dato: {df['P558E3'].dtype}")
    print(f"  • Valores únicos: {sorted(df['P558E3'].unique())}")
    print(f"  • Value counts:")
    print(df['P558E3'].value_counts(dropna=False).sort_index())
    
    # Convertir a numérico para estadísticas
    df['P558E3_num'] = pd.to_numeric(df['P558E3'], errors='coerce')
    print(f"\n  • Tasa de missing: {df['P558E3_num'].isna().mean():.2%}")
    print(f"  • Tasa de 'Sí' (valor 1): {(df['P558E3_num'] == 1).mean():.2%}")
    print(f"  • Tasa de 'No' (valor 2): {(df['P558E3_num'] == 2).mean():.2%}")
else:
    print("\n❌ P558E3 NO está en la base.")
    print("   → Opciones:")
    print("      a) Agregarla al script de construcción (recomendado)")
    print("      b) Eliminar la afirmación sobre denegación del documento")

# ============================================================
# 4. RECONCILIAR N (C1)
# ============================================================

print("\n" + "="*70)
print("3. RECONCILIAR N (C1)")
print("="*70)

print(f"\n📊 Total en base: {len(df):,}")

# Desagregación por grupos (Tabla A6)
informal_con = ((df['trabajador_informal'] == 1) & (df['usa_billetera'] == 1)).sum()
informal_sin = ((df['trabajador_informal'] == 1) & (df['usa_billetera'] == 0)).sum()
formal_con = ((df['trabajador_informal'] == 0) & (df['usa_billetera'] == 1)).sum()
formal_sin = ((df['trabajador_informal'] == 0) & (df['usa_billetera'] == 0)).sum()

print(f"\n📊 Desagregación por grupos:")
print(f"  • Informales, usa_billetera=1: {informal_con:,}")
print(f"  • Informales, usa_billetera=0: {informal_sin:,}")
print(f"  • Formales, usa_billetera=1: {formal_con:,}")
print(f"  • Formales, usa_billetera=0: {formal_sin:,}")
print(f"  • TOTAL SUMA: {informal_con + informal_sin + formal_con + formal_sin:,}")

if informal_con + informal_sin + formal_con + formal_sin == len(df):
    print("  ✅ Los N coinciden con el total de la base")
else:
    print(f"  ⚠️ Diferencia de {len(df) - (informal_con + informal_sin + formal_con + formal_sin):,} observaciones")

# Verificar si hay missing en las variables de grupo
missing_informal = df['trabajador_informal'].isna().sum()
missing_usa = df['usa_billetera'].isna().sum()
print(f"\n📊 Missing en variables de grupo:")
print(f"  • Missing en trabajador_informal: {missing_informal:,}")
print(f"  • Missing en usa_billetera: {missing_usa:,}")

# ============================================================
# 5. VERIFICAR TABLA A8 (IC y Wald)
# ============================================================

print("\n" + "="*70)
print("4. VERIFICAR TABLA A8 (C1)")
print("="*70)

# Simular los coeficientes de tu modelo (tomados de tu output)
coef_usa = 0.1938
se_usa = 0.1272

coef_informal = -0.2221
se_informal = 0.0954

coef_interaccion = -0.0782
se_interaccion = 0.1715

coef_educ = 0.0925
se_educ = 0.0192

print("\n📊 Intervalos de confianza CORRECTOS:")
print(f"  • usa_billetera: {coef_usa:.4f} ± {1.96*se_usa:.4f} → [{coef_usa - 1.96*se_usa:.4f}, {coef_usa + 1.96*se_usa:.4f}]")
print(f"  • trabajador_informal: {coef_informal:.4f} ± {1.96*se_informal:.4f} → [{coef_informal - 1.96*se_informal:.4f}, {coef_informal + 1.96*se_informal:.4f}]")
print(f"  • interacción: {coef_interaccion:.4f} ± {1.96*se_interaccion:.4f} → [{coef_interaccion - 1.96*se_interaccion:.4f}, {coef_interaccion + 1.96*se_interaccion:.4f}]")
print(f"  • nivel_educativo: {coef_educ:.4f} ± {1.96*se_educ:.4f} → [{coef_educ - 1.96*se_educ:.4f}, {coef_educ + 1.96*se_educ:.4f}]")

print(f"\n⚠️ En tu Tabla A8 actual, el IC de nivel_educativo es [-0.055, 0.130]")
print(f"   → El IC CORRECTO es [0.0549, 0.1301] (AMBOS límites positivos)")
print(f"\n⚠️ En tu Tabla A8 actual, Wald χ² = 185.53.45")
print(f"   → El Wald CORRECTO es 185.53")

# ============================================================
# 6. VERIFICAR NUMERACIÓN DE TABLAS
# ============================================================

print("\n" + "="*70)
print("5. NUMERACIÓN DE TABLAS (C1)")
print("="*70)

print("\n📊 Tablas que deberían existir:")
tablas_esperadas = ['A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8', 'A9', 'A10', 'A11', 'A12', 'A13', 'A14', 'A15', 'A16', 'A17', 'A18']
print(f"  • Tablas esperadas: {', '.join(tablas_esperadas)}")

print("\n📊 En tu índice de anexos aparece:")
print("  • A1, A2, A3, A4, A5, A6, A7, A8, A9, A10, A12, A13, A14, A15, A16, A17, A18")
print("  ❌ FALTA: A11")
print("  → Opciones:")
print("     a) Renombrar A12 → A11, A13 → A12, ... (y actualizar referencias)")
print("     b) Eliminar la referencia a A11 si no se usa")

# ============================================================
# 7. PREVALENCIA DEL TRATAMIENTO (C4)
# ============================================================

print("\n" + "="*70)
print("6. PREVALENCIA DEL TRATAMIENTO (C4)")
print("="*70)

tasa_uso = df['usa_billetera'].mean()
tasa_tenencia = df['tiene_billetera'].mean()
informalidad = df['trabajador_informal'].mean()

print(f"\n📊 En la muestra analítica:")
print(f"  • Tasa de USO de billetera: {tasa_uso:.1%} ({df['usa_billetera'].sum():,} usuarios)")
print(f"  • Tasa de TENENCIA de billetera: {tasa_tenencia:.1%} ({df['tiene_billetera'].sum():,} usuarios)")
print(f"  • Tasa de informalidad: {informalidad:.1%}")

print(f"\n📊 A nivel nacional (Credicorp, 2025):")
print("  • Tenencia de billetera: 58%")

print(f"\n📌 Diferencia explicada:")
print("  • 58% = Tenencia nacional en población adulta (tiene Yape/Plin)")
print("  • 13.7% = Uso efectivo en la muestra analítica (usa como medio de pago)")
print("  • Universo diferente: jefes de hogar ocupados sin crédito formal en 2024")

# ============================================================
# 8. RESUMEN FINAL - ¿QUÉ HACER?
# ============================================================

print("\n" + "="*70)
print("7. RESUMEN - ¿QUÉ HACER?")
print("="*70)

print("\n📋 DECISIONES A TOMAR:")

# Decisión 1: P558E2 y P558E3
if 'P558E2' in df.columns and 'P558E3' in df.columns:
    print("\n  ✅ P558E2 y P558E3 YA ESTÁN en la base.")
    print("     → Solo necesitas generar la tabla descriptiva para el documento.")
else:
    print("\n  ❌ P558E2 y/o P558E3 NO están en la base.")
    print("     → Opciones:")
    print("        a) AGREGARLAS al script de construcción (recomendado)")
    print("        b) ELIMINAR la afirmación incorrecta del documento")
    print("        c) MANTENER la afirmación como limitación (pero corregir el texto)")

# Decisión 2: Tabla A11
print("\n  ❌ Tabla A11 no existe.")
print("     → Opciones:")
print("        a) Renumerar: A12→A11, A13→A12, ...")
print("        b) Eliminar referencia a A11")

# Decisión 3: IC de nivel educativo
print("\n  ❌ IC de nivel educativo incorrecto en Tabla A8.")
print("     → CORREGIR: [0.0549, 0.1301]")

# Decisión 4: Wald χ²
print("\n  ❌ Wald χ² incorrecto en Tabla A8.")
print("     → CORREGIR: 185.53 (sin .45)")

# Decisión 5: C4
print("\n  ❌ Falta explicación de 13.7% vs 58%.")
print("     → AGREGAR en sección de descriptivos")

print("\n" + "="*70)
print("✅ AUDITORÍA COMPLETADA")
print("="*70)

AUDITORÍA FASE 1 - BASE_REGRESIONES.csv
✅ Base cargada: 6,358 observaciones, 18 variables
✅ Columnas disponibles: ['id_persona', 'jefe_hogar', 'sexo', 'edad', 'ocupado', 'trabajador_informal', 'tiene_billetera', 'usa_billetera', 'credito_formal_2024', 'factor_expansion', 'nivel_educativo', 'estrato', 'dominio', 'miembros_hogar', 'conglomerado', 'credito_formal_2025', 'credito_informal_2025', 'nuevo_credito_formal']

1. VARIABLES FINANCIERAS DISPONIBLES

📊 Columnas financieras encontradas:
  • credito_formal_2024
  • credito_formal_2025
  • credito_informal_2025
  • nuevo_credito_formal

📊 Buscando P558E2 y P558E3 específicamente:
  ❌ P558E2 NO encontrada en la base
  ❌ P558E3 NO encontrada en la base

2. ANÁLISIS DE P558E2 Y P558E3 (SI EXISTEN)

❌ P558E2 NO está en la base.
   → Opciones:
      a) Agregarla al script de construcción (recomendado)
      b) Eliminar la afirmación sobre solicitud del documento

❌ P558E3 NO está en la base.
   → Opciones:
      a) Agregarla al script de co

Problema	Estado	Qué hacer
P558E2 y P558E3	❌ No están en la base	Agregarlas al script de construcción (tienes el código, solo falta añadirlas)
N en Tabla A6	✅ Coinciden (6,358)	Nada que hacer
IC de nivel educativo	❌ Incorrecto en Tabla A8	Corregir en el documento
Wald χ²	❌ Incorrecto en Tabla A8	Corregir en el documento
Tabla A11	❌ No existe	Renumerar o eliminar referencia
Explicación 13.7% vs 58%	❌ Falta	Agregar en sección de descriptivos